In [ ]:
!pip install pymilvus sentence-transformers google-genai

import os
import json
import time
import pandas as pd
from sentence_transformers import SentenceTransformer
from pymilvus import MilvusClient
from google import genai

# --- CONFIGURATION ---
# Using your provided keys and paths
os.environ['GOOGLE_API_KEY'] = 'API_KEY'
client_genai = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])

  Using cached pymilvus-2.6.12-py3-none-any.whl.metadata (6.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.2/315.2 kB 14.0 MB/s eta 0:00:00


In [ ]:
!pip install "pymilvus[milvus_lite]" sentence-transformers google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 20.3 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata, drive
import shutil # Import the shutil module
drive.mount('/content/drive')

# Updated Paths
BASE_DIR = "PATH TO BASE DIR"
TEST_FOLDER = f"{BASE_DIR}/test"
DB_PATH = f"{TEST_FOLDER}/milvus_demo1.db" # Pointing to your test folder
LOCAL_DB_PATH = "/content/milvus_baseline.db" # Fast local storage
TEST_FILE = f"{TEST_FOLDER}/jurismind_100.csv"
BASELINE_RESULTS_FILE = f"{TEST_FOLDER}/standard_rag_results_scored.csv"

# 2. LOCAL COPY (Fixes the Connection Error)
if not os.path.exists(LOCAL_DB_PATH):
    print(f"Copying Milvus DB to local storage for performance...")
    shutil.copy(DB_PATH, LOCAL_DB_PATH)
    print("Copy complete.")

milvus_client = MilvusClient(DB_PATH)
coll_name = "legalverse"
model_emb = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# --- GLOBAL CLIENT HANDLER ---
_milvus_client = None

def get_milvus_client():
    global _milvus_client
    if _milvus_client is None:
        _milvus_client = MilvusClient(LOCAL_DB_PATH)
    return _milvus_client

In [ ]:
# --- STANDARD RAG PIPELINE ---
def run_standard_rag_pipeline(query, topk=5):
    client = get_milvus_client()
    query_vector = model_emb.encode([query])

    # Attempt search with one automatic retry if the channel is closed
    try:
        search_results = client.search(
            collection_name=coll_name,
            data=query_vector,
            limit=topk,
            output_fields=["text"],
        )
    except Exception as e:
        return f"RETRIEVAL ERROR: {str(e)}"

    context_chunks = [x['entity']['text'] for x in search_results[0]] if search_results else []
    concatenated_context = "\n\n".join(context_chunks)

    base_prompt = f"""
    You are a standard legal assistant.
    Answer the user query using ONLY the provided context items.

    STRICT RULES:
    1. If the answer is not explicitly contained within the context, state: "I cannot find the answer to this query in the provided source content."
    2. Do NOT use any external or internal legal knowledge.

    CONTEXT:
    {concatenated_context}

    USER QUERY: {query}
    """

    try:
        response = client_genai.models.generate_content(model="gemini-flash-latest", contents=base_prompt)
        return response.text
    except Exception as e:
        return f"LLM ERROR: {str(e)}"

In [ ]:
# --- THE JUDGE AGENT ---
def judge_score(query, gt_answer, ai_response):
    """Evaluates Standard RAG response against Ground Truth."""
    judge_prompt = f"""
    You are a Fair Legal Evaluator. Compare the AI's response to the Ground Truth.

    GROUND TRUTH:
    {gt_answer}

    AI RESPONSE:
    {ai_response}

    SCORING CRITERIA (0-100):
    1. Accuracy: Does the answer match the ground truth factually?
    2. Quality: Is the formatting clear?

    NOTE: Do not be overly strict on minor linguistic variations. Focus on factual correctness.
    Return a JSON object: {{"score": 85, "reason": "..."}}
    """
    try:
        resp = client_genai.models.generate_content(model="gemini-flash-latest", contents=judge_prompt)
        # Clean markdown if present
        clean_resp = resp.text.replace("```json", "").replace("```", "").strip()
        return json.loads(clean_resp)
    except:
        return {"score": 0, "reason": "Judge Error"}


In [ ]:
# --- MAIN EVALUATION LOOP ---
def run_baseline_evaluation():
    if os.path.exists(BASELINE_RESULTS_FILE):
        df = pd.read_csv(BASELINE_RESULTS_FILE)
        # Clear previous "RETRIEVAL ERROR" rows to retry them locally
        df.loc[df['Baseline_RAG_Response'].str.contains('RETRIEVAL ERROR', na=False), ['Baseline_RAG_Response', 'Baseline_RAG_Score']] = None
    else:
        df = pd.read_csv(TEST_FILE)
        df['Baseline_RAG_Response'] = None
        df['Baseline_RAG_Score'] = None
        df['Baseline_RAG_Band'] = None

    print(f"Starting Baseline RAG evaluation for {len(df)} questions...")

    try:
        for index, row in df.iterrows():
            if pd.notna(row['Baseline_RAG_Score']): continue

            ai_response = run_standard_rag_pipeline(row['Question'])
            judgment = judge_score(row['Question'], row['Ground_Truth_Answer'], ai_response)

            score = judgment.get('score', 0)
            df.at[index, 'Baseline_RAG_Response'] = ai_response
            df.at[index, 'Baseline_RAG_Score'] = score
            df.at[index, 'Baseline_RAG_Band'] = "High" if score >= 80 else "Medium" if score >= 40 else "Low"

            df.to_csv(BASELINE_RESULTS_FILE, index=False)
            print(f" [{index+1}/100] Baseline Score: {score} | 💾 Saved")
            time.sleep(12)
    finally:
        global _milvus_client
        if _milvus_client: _milvus_client.close()

In [ ]:
# Using your provided keys and paths
os.environ['GOOGLE_API_KEY'] = 'API_KEY'
client_genai = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])

In [ ]:
run_baseline_evaluation()